# Domain-based raw HTML crawler

Load the ViBioMIR corpus, split it between worker accounts, render each URL with Playwright, and save the raw HTML. This notebook does not use Trafilatura or perform content extraction.

In [ ]:
from pathlib import Path

from datasets import load_dataset

# Zero-based worker/account ID and total number of accounts.
ID = 0
NUM_ACCOUNT = 4

if NUM_ACCOUNT <= 0:
    raise ValueError("NUM_ACCOUNT must be greater than 0")
if not 0 <= ID < NUM_ACCOUNT:
    raise ValueError("ID must be between 0 and NUM_ACCOUNT - 1")

OUTPUT_DIR = Path("output") / "raw_html" / f"id_{ID}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "id.txt").write_text(f"{ID}\n", encoding="utf-8")

ds = load_dataset("AIGuruTinix/ViBioMIR", "corpus")


In [ ]:
%pip install playwright
!playwright install-deps chromium
!playwright install chromium


In [ ]:
import asyncio
from urllib.parse import urlparse

from playwright.async_api import (
    Error as PlaywrightError,
    TimeoutError as PlaywrightTimeoutError,
    async_playwright,
)


async def fetch_html(url: str) -> dict:
    """Render a URL with Playwright and return its raw HTML."""
    async with async_playwright() as playwright:
        browser = await playwright.chromium.launch(headless=True)
        page = await browser.new_page()
        page.set_default_timeout(15_000)
        response = None

        try:
            # Wait only for navigation to commit so third-party resources do not
            # keep the crawler blocked. The later waits allow client-side rendering.
            try:
                response = await page.goto(
                    url,
                    wait_until="commit",
                    timeout=30_000,
                )
            except PlaywrightTimeoutError:
                pass

            try:
                await page.wait_for_selector(
                    "body",
                    state="attached",
                    timeout=10_000,
                )
            except PlaywrightTimeoutError:
                pass

            await asyncio.sleep(2)

            try:
                await page.wait_for_load_state(
                    "domcontentloaded",
                    timeout=10_000,
                )
            except PlaywrightTimeoutError:
                pass

            html = None
            last_error = None

            # page.content() can fail if a redirect starts while the DOM is read.
            for _ in range(3):
                try:
                    html = await page.content()
                    break
                except PlaywrightError as error:
                    last_error = error
                    await asyncio.sleep(1)

            if html is None:
                raise RuntimeError(
                    f"Unable to retrieve stable page content: {last_error}"
                )

            final_url = page.url
            return {
                "url": url,
                "final_url": final_url,
                "domain": urlparse(final_url).netloc,
                "status_code": response.status if response else None,
                "html": html,
                "success": True,
            }

        except Exception as error:
            return {
                "url": url,
                "domain": urlparse(url).netloc,
                "success": False,
                "error": str(error),
            }

        finally:
            await browser.close()


In [ ]:
total_rows = len(ds["train"])
start = total_rows * ID // NUM_ACCOUNT
end = total_rows * (ID + 1) // NUM_ACCOUNT
urls = [ds["train"][i] for i in range(start, end)]

print(f"ID {ID}/{NUM_ACCOUNT - 1}: processing rows {start}:{end} ({len(urls)} rows)")


In [ ]:
urls[0] if urls else None


In [ ]:
import pandas as pd
from tqdm.auto import tqdm

rows = []
error_urls = []

for item in tqdm(urls, desc=f"Crawling raw HTML for ID {ID}"):
    doc_id = item["id"]
    url = item["url"]
    result = await fetch_html(url)

    if not result["success"]:
        print(f"Error at {url}: {result['error']}")
        error_urls.append(
            {"doc_id": doc_id, "url": url, "error": result["error"]}
        )
        continue

    result["doc_id"] = doc_id
    rows.append(result)

raw_html_df = pd.DataFrame(
    rows,
    columns=[
        "doc_id",
        "url",
        "final_url",
        "domain",
        "status_code",
        "html",
        "success",
    ],
)


In [ ]:
raw_html_df.head()


In [ ]:
df_errors = pd.DataFrame(error_urls, columns=["doc_id", "url", "error"])
df_errors.to_csv(OUTPUT_DIR / "failed_urls.csv", index=False, encoding="utf-8-sig")
print(f"Saved {len(df_errors)} failed URLs to {OUTPUT_DIR / 'failed_urls.csv'}")


In [ ]:
raw_html_df.to_csv(OUTPUT_DIR / "successful_urls.csv", index=False, encoding="utf-8-sig")
print(
    f"Saved {len(raw_html_df)} raw HTML documents to "
    f"{OUTPUT_DIR / 'successful_urls.csv'}"
)
